# Deciphering Diabetes - Project Code

This notebook contains the analysis of diabetes data including CDC diabetes statistics by state and hospital diabetic patient data.

## Import Libraries

Importing necessary libraries for data analysis and visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
sns.set_style('darkgrid')

## Data Processing Functions and Loading

Defining helper functions and loading the CDC diabetes data by state.

In [ ]:
# Mapping between state names and abbreviations
us_state_to_abbrev = {
    "Alabama": "AL",
    "Alaska": "AK",
    "Arizona": "AZ",
    "Arkansas": "AR",
    "California": "CA",
    "Colorado": "CO",
    "Connecticut": "CT",
    "Delaware": "DE",
    "Florida": "FL",
    "Georgia": "GA",
    "Hawaii": "HI",
    "Idaho": "ID",
    "Illinois": "IL",
    "Indiana": "IN",
    "Iowa": "IA",
    "Kansas": "KS",
    "Kentucky": "KY",
    "Louisiana": "LA",
    "Maine": "ME",
    "Maryland": "MD",
    "Massachusetts": "MA",
    "Michigan": "MI",
    "Minnesota": "MN",
    "Mississippi": "MS",
    "Missouri": "MO",
    "Montana": "MT",
    "Nebraska": "NE",
    "Nevada": "NV",
    "New Hampshire": "NH",
    "New Jersey": "NJ",
    "New Mexico": "NM",
    "New York": "NY",
    "North Carolina": "NC",
    "North Dakota": "ND",
    "Ohio": "OH",
    "Oklahoma": "OK",
    "Oregon": "OR",
    "Pennsylvania": "PA",
    "Rhode Island": "RI",
    "South Carolina": "SC",
    "South Dakota": "SD",
    "Tennessee": "TN",
    "Texas": "TX",
    "Utah": "UT",
    "Vermont": "VT",
    "Virginia": "VA",
    "Washington": "WA",
    "West Virginia": "WV",
    "Wisconsin": "WI",
    "Wyoming": "WY",
    "District of Columbia": "DC",
    "American Samoa": "AS",
    "Guam": "GU",
    "Northern Mariana Islands": "MP",
    "Puerto Rico": "PR",
    "United States Minor Outlying Islands": "UM",
    "Virgin Islands of the U.S.": "VI",
    "Median of States": "MED"
}
# Process CDC dataframe - create numeric columns and generate state abbreviations
def processCDCDataFrame(df: pd.DataFrame):
    df['Percentage'] = pd.to_numeric(df.Percentage, errors='coerce')
    df['Lower Limit'] = pd.to_numeric(df['Lower Limit'], errors='coerce')
    df['Upper Limit'] = pd.to_numeric(df['Upper Limit'], errors='coerce')
    df['StateAbbr'] = df['State'].map(us_state_to_abbrev)
    return df
# Concat multiple dataframes with a column containing categories for each constituent dataframe
def joinDfs(dfs: list[pd.DataFrame], columnName: str, keys: list[str]):
    return pd.concat(dfs, keys=keys).reset_index(level=0, names=columnName).reset_index(drop=True)
# Loading raw CSV files
statesOverYearsDf = pd.read_csv('DiabetesAtlas_AllStatesLineChartData.csv')
stateEduAdult = pd.read_csv('DiabetesAtlas_States_Edu_Adult.csv')
stateEduHighSchool = pd.read_csv('DiabetesAtlas_States_Edu_HighSchool.csv')
stateEduKid = pd.read_csv('DiabetesAtlas_States_Edu_Kid.csv')
stateAgeYA = pd.read_csv('DiabetesAtlas_States_Age_YA.csv')
stateAgeAdult = pd.read_csv('DiabetesAtlas_States_Age_Adult.csv')
stateAgeRetiree = pd.read_csv('DiabetesAtlas_States_Age_Retirees.csv')
stateAgeElderly = pd.read_csv('DiabetesAtlas_States_Age_Elderly.csv')
# Processing raw CSV files and build final dataframe
statesOverYearsDf = processCDCDataFrame(statesOverYearsDf)
stateEduAdult = processCDCDataFrame(stateEduAdult)
stateEduHighSchool = processCDCDataFrame(stateEduHighSchool)
stateEduKid = processCDCDataFrame(stateEduKid)
stateAgeYA = processCDCDataFrame(stateAgeYA)
stateAgeAdult = processCDCDataFrame(stateAgeAdult)
stateAgeRetiree = processCDCDataFrame(stateAgeRetiree)
stateAgeElderly = processCDCDataFrame(stateAgeElderly)
stateAgeDf = joinDfs(
    [stateAgeYA, stateAgeAdult, stateAgeRetiree, stateAgeElderly],
    'AgeLevel',
    keys=['YA', 'Adult', 'Retirees', 'Elderly']
)
stateEduDf = joinDfs(
    [stateEduAdult, stateEduHighSchool, stateEduKid],
    'EducationLevel',
    keys=['Upper Level Study', 'High School Diploma', 'No High School']
)

In [ ]:
statesOverYearsDf.describe()

## Hospital Diabetic Patient Data

Loading and processing the hospital diabetic patient dataset.

In [ ]:
diabeticDataDf = pd.read_csv('diabetic_data.csv')
diabeticDataDf.head()

In [ ]:
# Mapping of variables to binned variables of interest and values
diabeticDataDf.admission_type_id = diabeticDataDf.admission_type_id.map({
    1: 'Emergency',
    2: 'Emergency',
    7: 'Emergency',
    3: 'Elective',
    4: 'Elective'
})
diabeticDataDf.discharge_disposition_id = diabeticDataDf.discharge_disposition_id.map({
    2: 'Transferred to another medical facility',
    5: 'Transferred to another medical facility',
    14: 'Transferred to another medical facility',
    23: 'Transferred to another medical facility',
    24: 'Transferred to another medical facility',
    27: 'Transferred to another medical facility',
    28: 'Transferred to another medical facility',
    29: 'Transferred to another medical facility',
    30: 'Transferred to another medical facility',
    7: 'Left against medical advice',
    9: 'In hospital',
    12: 'In hospital',
    11: 'Expired in Hospital',
    19: 'Expired out of Hospital',
    20: 'Expired out of Hospital',
    21: 'Expired out of Hospital',
})
diabeticDataDf.admission_source_id = diabeticDataDf.admission_source_id.map({
    1: 'Physician Referral',
    2: 'Physician Referral',
    3: 'Physician Referral',
    4: 'Hospital Transfer',
    5: 'Skilled Nursing Facility Transfer',
    6: 'Other Healthcare Facility Transfer',
    10: 'Hospital Transfer',
    22: 'Hospital Transfer',
    25: 'Hospital Transfer',
    7: 'Emergency Room'
})
diabeticDataDf.max_glu_serum = diabeticDataDf.max_glu_serum.replace(['None'], np.nan)
diabeticDataDf.A1Cresult = diabeticDataDf.A1Cresult.replace(['None'], np.nan)
diabeticDataDf = diabeticDataDf.loc[:, [
                                           'race', 'gender', 'age',
                                           'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
                                           'time_in_hospital', 'num_lab_procedures', 'num_procedures',
                                           'num_medications',
                                           'number_outpatient', 'number_emergency', 'number_inpatient',
                                           'number_diagnoses',
                                           'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
                                           'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
                                           'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
                                           'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin',
                                           'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone',
                                           'metformin-pioglitazone', 'max_glu_serum', 'A1Cresult',
                                           'diabetesMed', 'readmitted'
                                       ]]

In [ ]:
diabeticDataDf.describe()

## Exploratory Data Analysis

Analyzing the distribution and characteristics of the datasets.

In [ ]:
# Distribtion of categorical variables
def describeCategorical(x: pd.Series):
    return pd.Series({
        'Missing Values': x.isna().sum(),
        'Unique Values': x.nunique(),
        'Top Levels': str(x.value_counts().nlargest(3).to_dict())
    })
diabeticDataDf.select_dtypes('object').apply(describeCategorical)

In [ ]:
# Calculate diabetes rate of change
statesOverYearsDf['PercentageDiff'] = statesOverYearsDf.groupby('StateAbbr')['Percentage'].diff()
stateEduDf['PercentageDiff'] = stateEduDf.groupby(['StateAbbr', 'EducationLevel'])['Percentage'].diff()
stateAgeDf['PercentageDiff'] = stateAgeDf.groupby(['StateAbbr', 'AgeLevel'])['Percentage'].diff()

## Diabetes Trends Analysis

Analyzing trends in diabetes rates over time and by state.

In [ ]:
g = sns.relplot(statesOverYearsDf[statesOverYearsDf.State == 'Median of States'],
                x='Year', y='Percentage',
                lw=4,
                kind='line', aspect=2)
g.ax.set_title('US Median Diabetes Percentage')
g.set_xlabels('')
g.ax.set_ylim(4.5, 10)
plt.twinx()
ax = sns.lineplot(statesOverYearsDf[statesOverYearsDf.State == 'Median of States'],
                  x='Year', y='PercentageDiff',
                  lw=4, ls='--', color='orange')
ax.grid(False)
ax.set_ylabel("YoY Δ Percentage")
ax.set_ylim(-1, 3)
ax.tick_params(axis='y', length=0)

In [ ]:
# Calculate average rate of change, as well as states with an average rate of change > 0.25
meanStateDiff = statesOverYearsDf.groupby('StateAbbr')['PercentageDiff'].mean().reset_index()
meanMedianDiff = meanStateDiff[meanStateDiff.StateAbbr == 'MED']['PercentageDiff']
highStateDiff = meanStateDiff[meanStateDiff['PercentageDiff'] > 0.25].copy()
meanStateDiff.loc[meanStateDiff.StateAbbr == 'MED', 'StateAbbr'] = '~'
g = sns.catplot(meanStateDiff,
                x='StateAbbr', y='PercentageDiff', hue='PercentageDiff',
                aspect=3, legend_out=False)
g.set_xlabels('State')
g.set_ylabels('Average Yearly Change')
g.ax.plot('~', meanMedianDiff, 'o', markersize=12)
g.ax.text('~', meanMedianDiff + 0.01, 'Median', ha='center', va='bottom', weight='semibold')
g.ax.axhline(0.25, ls='-.')
g.ax.plot(highStateDiff.StateAbbr, highStateDiff.PercentageDiff, 'o', color='orange', markersize=10)
for i, row in highStateDiff.iterrows():
    g.ax.text(row['StateAbbr'], row['PercentageDiff'], f"  {row['StateAbbr']}",
              ha='left', va='center_baseline', color='black', weight='semibold')
g.ax.margins(x=0.01)
sns.move_legend(g.ax, 'center left', bbox_to_anchor=(1, 0.5), title='Δ Percentage')

## Demographic Analysis

Analyzing patient demographics and their relationship to diabetes outcomes.

In [ ]:
# Select pertinent columns for analysis
nomedData = diabeticDataDf.loc[:, ['race', 'gender', 'age', 'discharge_disposition_id',
                                   'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications',
                                   'number_emergency', 'number_diagnoses'
                                   ]].copy()
nomedData = nomedData[nomedData.gender != 'Unknown/Invalid']
nomedData = nomedData[nomedData.race != '?']
nomedData.head()

In [ ]:
plot1 = sns.catplot(nomedData, x='age', y='number_diagnoses', aspect=2, kind='bar', color='lightblue')
plot1.set_axis_labels("Age Groups", "Number of Diagnoses")
plot1.fig.suptitle("Number of Diagnoses by Age", y=1.05)

In [ ]:
plot2 = sns.catplot(nomedData, x='age', y='time_in_hospital', aspect=2, kind='bar', color='lightblue')
plot2.set_axis_labels("Age Groups", "Time in Hospital")
plot2.fig.suptitle("Time in Hospital by Age", y=1.05);

In [ ]:
nomedData['num_total_procedures'] = nomedData.num_lab_procedures + nomedData.num_procedures
#create a column that combines both lab and other (including surgical) procedures
plot3 = sns.catplot(nomedData, x='age', y='num_total_procedures', aspect=2, kind='bar', color='lightblue')
plot3.set_axis_labels("Age Groups", "Number of Total Procedures")
plot3.fig.suptitle("Number of Total Procedures by Age", y=1.05)
mean_lab_procedures = nomedData.groupby('age')['num_lab_procedures'].mean().round().astype(int)
#these are the mean lab procedures across all age groups
print(mean_lab_procedures)

In [ ]:
custom_palette = {'Female': '#b19cd9', 'Male': 'skyblue'}
plotorder = nomedData.groupby('race')['number_emergency'].mean().sort_values().index
g = sns.catplot(nomedData, x='race', hue='gender', y='number_emergency',
                errorbar='se',
                order=plotorder,
                aspect=2, kind='bar', palette=custom_palette)
g.ax.axhline(0.205, ls='-.')
g.set_axis_labels("Race", "Number of Emergency Visits Per Year")
g.set_titles("Number of Emergency Visits by Racial Groups")
g.fig.suptitle("Number of Emergency Visits Per Year by Racial Group and Gender", y=1.05);

In [ ]:
nomedData.groupby(['race', 'gender'])['number_emergency'].mean()

In [ ]:
gender_emergency = nomedData.groupby('gender')['number_emergency'].mean().reset_index()
print(gender_emergency)
print('0.219673/0.179955 = 1.2207107332388654')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(10, 5))
# Plot each categorical plot on its own subplot
sns.barplot(data=nomedData, x='gender', y='number_diagnoses', hue='gender', ax=axes[0], palette=custom_palette,
            legend=False)
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Number of Diagnoses')
sns.barplot(data=nomedData, x='gender', y='num_total_procedures', hue='gender', ax=axes[1], palette=custom_palette,
            legend=False)
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Number of Total Procedures')
sns.barplot(data=nomedData, x='gender', y='time_in_hospital', hue='gender', ax=axes[2], palette=custom_palette,
            legend=False)
axes[2].set_xlabel('Gender')
axes[2].set_ylabel('Time in Hospital')
sns.barplot(data=nomedData, x='gender', y='num_medications', hue='gender', ax=axes[3], palette=custom_palette,
            legend=False)
axes[3].set_xlabel('Gender')
axes[3].set_ylabel('Number of Medications')
plt.suptitle('Analysis by Gender', y=0.95);
plt.tight_layout()

In [ ]:
print(nomedData.groupby('gender')['number_diagnoses'].mean().reset_index())
print(nomedData.groupby('gender')['num_total_procedures'].mean().reset_index())
print(nomedData.groupby('gender')['time_in_hospital'].mean().reset_index())
print(nomedData.groupby('gender')['num_medications'].mean().reset_index())

In [ ]:
race_emergency = nomedData.groupby('race')['number_emergency'].mean().reset_index()
print(race_emergency)
asian = 0.093604 / 0.185679
#Asians are this much less likely to go to the emergency room than Caucasians
print('Asian/Caucasian ratio =', asian)
AfricanAmerican = 0.261010 / 0.185679
print('African American/Caucasian ratio =', AfricanAmerican)
Hispanic = 0.228277 / 0.185679
print('Hispanic/Caucasian ratio =', Hispanic)

## Hospital Outcomes Analysis

Analyzing patient outcomes including mortality and transfer rates.

In [ ]:
dischargedData = nomedData.copy().dropna()
#drop irrelevant mappings after we labeled relevant discharge outcomes (expired or transfers)
dischargedData['expired'] = dischargedData.discharge_disposition_id.str.contains('Expired').map(
    {True: 'Expired', False: 'Other Outcomes'})
#create a column that shows whether they expired or not
race_ct = pd.crosstab(dischargedData['race'], dischargedData['expired'], normalize='index')
#normalize the index to compare within different racial groups to see if the ratio of expired within each population is different
race_plot = race_ct.plot.bar()  # Adjust the number of colors
race_plot.set_title('Death of Diabetic Patients by Racial Group')
race_plot.set_xlabel('Race')
race_plot.set_ylabel('Proportion of Expired')
race_plot.set_xticklabels(race_plot.get_xticklabels(), rotation=45)
plt.legend(['Expired', 'Other Outcomes'], bbox_to_anchor=(1.05, 1), loc='upper left')
print(pd.crosstab(dischargedData['race'], dischargedData['expired'], normalize='index').round(2))

In [ ]:
dischargedData['transferred'] = (
    dischargedData.discharge_disposition_id.isin(['Transferred to another medical facility' or 'In hospital'])).map(
    {True: 'To a medical facility', False: 'Other outcomes'})
#creating a column of whether or not a patient was transferred to another medical facility for continued care
race_ct = pd.crosstab(dischargedData['race'], dischargedData['transferred'], normalize='index')
race_plot = race_ct.plot.bar()  # Adjust the number of colors
race_plot.set_title('Transfers of Diabetic Patients by Racial Group')
race_plot.set_xlabel('Race')
race_plot.set_ylabel('Proportion of Transfers')
race_plot.set_xticklabels(race_plot.get_xticklabels(), rotation=45)
plt.legend(['Other Outcomes', 'To a medical facility'], bbox_to_anchor=(1.05, 1), loc='upper left')
print(pd.crosstab(dischargedData['race'], dischargedData['transferred'], normalize='index').round(2))

In [ ]:
# creating a subset from the dataset with relevant columns
nomedData1 = diabeticDataDf.loc[:, ['age', 'discharge_disposition_id', 'admission_source_id', 'number_inpatient',
                                    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
                                    ]].copy()

In [ ]:
nomedData1['num_total_procedures'] = nomedData1['num_procedures'] + nomedData1['num_lab_procedures']

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 5))
sns.barplot(data=nomedData1, x='admission_source_id', y='number_inpatient', ax=axes[0],
            palette=['#728FCE', '#9E7BFF', '#FC6C85', '#90EE90', '#F4A460'])
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=50, ha='right')
axes[0].set_xlabel('Admission Source')
axes[0].set_ylabel('number_inpatient')
sns.barplot(data=nomedData1, x='admission_source_id', y='num_total_procedures',
            palette=['#728FCE', '#9E7BFF', '#FC6C85', '#90EE90', '#F4A460'], ax=axes[1])
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=50, ha='right')
axes[1].set_xlabel('Admission Source')
axes[1].set_ylabel('num_total_procedures')
sns.barplot(data=nomedData1, x='admission_source_id', y='time_in_hospital',
            palette=['#728FCE', '#9E7BFF', '#FC6C85', '#90EE90', '#F4A460'], ax=axes[2])
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=50, ha='right')
axes[2].set_xlabel('Admission Source')
axes[2].set_ylabel('time_in_hospital (day)')
plt.suptitle('Analysis by Admission Source')
plt.tight_layout()

In [ ]:
#creating dummy variables for the categorical variables in discharge disposition
expired = pd.get_dummies(nomedData1['discharge_disposition_id'])

In [ ]:
nomedData1['Expired in Hospital'] = expired['Expired in Hospital']
nomedData1['Expired out of Hospital'] = expired['Expired out of Hospital']

In [ ]:
# getting the proportion of expiration for patients outside of skilled nusing facilities
nomedData1[nomedData1['admission_source_id'] != 'Skilled Nursing Facility Transfer']['Expired in Hospital'].mean()

In [ ]:
# getting the proportion of expiration for patients from skilled nusing facilities
nomedData1[nomedData1['admission_source_id'] == 'Skilled Nursing Facility Transfer']['Expired in Hospital'].mean()

In [ ]:
#dividing the proportion of expiration for patients from skilled nursing facilities by the proportion of expiration for patients from other sources
0.047953216374269005 / 0.015593191434935996

## Drug Treatment Analysis

Analyzing drug treatments and their relationship to patient glucose levels.

In [ ]:
# sns.set(font_scale = 1)
sns.barplot(data=nomedData1, x='admission_source_id', y='Expired in Hospital',
            palette=['#728FCE', '#9E7BFF', '#FC6C85', '#90EE90', '#F4A460'])
plt.xticks(rotation=50, ha='right')
plt.ylabel('Expiration in Hospital')
plt.xlabel('Admission Source')
plt.suptitle('Proportion of Expiration Across Admission Sources')

In [ ]:
# filtering the data to exclude observations with less than ten total observartions by age and admission sources
filtered_data = nomedData1.groupby(['age', 'admission_source_id']).filter(lambda x: len(x) > 10)

In [ ]:
# sns.set(font_scale = 1)
sns.barplot(data=filtered_data, x='admission_source_id', y='Expired in Hospital', hue='age', ci=None)
plt.xticks(rotation=50, ha='right')
plt.ylabel('Expiration in Hospital')
plt.xlabel('Admission Source')
plt.suptitle('Proportion of Expiration Across Admission Sources')
plt.legend(title='Age Groups', loc='lower right', bbox_to_anchor=(1.25, 0));

In [ ]:
drug_columns = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
    'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
    'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin',
    'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone',
    'metformin-pioglitazone'
]
print(*drug_columns, sep=' ,')

In [ ]:
# Subsetting data where there are glucose values
glucose = diabeticDataDf[~diabeticDataDf.max_glu_serum.isnull()]
# plot of the number of occurances of each drug by glucose level
plt.figure(figsize=(10, 5))
# subsetting and formatting data for ploting
df_melted_glucose = pd.melt(glucose, id_vars=['max_glu_serum'], value_vars=drug_columns)
df_melted_glucose = df_melted_glucose[df_melted_glucose['value'] != 'No']
sns.countplot(data=df_melted_glucose, x='variable', hue='max_glu_serum', order=drug_columns,
              hue_order=['Norm', '>200', '>300'])
plt.title('Counts of Drug Treatments by Glucose Levels')
plt.xlabel('Drug')
plt.ylabel('Count')
plt.legend(title='Max Glucose Serum Test Result', loc='upper left')
plt.xticks(ticks=range(len(drug_columns)), labels=[drug.capitalize() for drug in drug_columns], rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Subsetting columns of 6 drugs of interest
drugs_of_interest_columns = ['metformin', 'glipizide', 'glyburide', 'pioglitazone', 'rosiglitazone', 'insulin']
# plot of the number of occurances of each of the 6 most common drugs by glucose level
plt.figure(figsize=(8, 5))
# Subsetting and formatting data for plotting
df_melted_glucose_interest = pd.melt(glucose, id_vars=['max_glu_serum'], value_vars=drugs_of_interest_columns)
df_melted_glucose_interest = df_melted_glucose_interest[df_melted_glucose_interest['value'] != 'No']
# plotting distribution of glucose levels
sns.countplot(data=df_melted_glucose_interest, x='variable', hue='max_glu_serum', order=drugs_of_interest_columns,
              hue_order=['Norm', '>200', '>300'])
plt.title('Counts of Select Drug Treatments by Glucose Levels')
plt.xlabel('Drug')
plt.ylabel('Count')
plt.legend(title='Max Glucose Serum Test Result', loc='upper left')
plt.xticks(ticks=range(len(drugs_of_interest_columns)),
           labels=[drug.capitalize() for drug in drugs_of_interest_columns], ha='center')
plt.show()

In [ ]:
# Calculate the proportions of glucose levels for each drug
proportion_data = df_melted_glucose_interest.groupby(['variable', 'max_glu_serum']).size().unstack()
proportion_data = proportion_data.divide(proportion_data.sum(axis=1), axis=0)
# Plotting proportion data
plt.figure(figsize=(8, 5))
# Settig the bottom of the bars
bottom = np.zeros(len(proportion_data))
# Iterate over each glucose level
for glu_level in ['Norm', '>200', '>300']:
    plt.bar(
        proportion_data.index,
        proportion_data[glu_level],
        bottom=bottom,
        label=glu_level
    )
    bottom += proportion_data[glu_level].values
plt.title('Proportions of Select Drug Treatments by Glucose Levels')
plt.xlabel('Drug')
plt.ylabel('Proportion')
plt.xticks(ticks=range(len(proportion_data.index)), labels=[drug.capitalize() for drug in proportion_data.index],
           ha='center')
plt.legend(title='Max Glucose Serum Test Result', bbox_to_anchor=(1, 1))
plt.show()

In [ ]:
# Subsetting data for Graph of Insulin and Metformin usage based of glucose levels
administeredMeds = diabeticDataDf[
    (~diabeticDataDf.metformin.isin(['No', 'Down'])) & (~diabeticDataDf.insulin.isin(['No', 'Down']))]
# Finding propotion of insulin usage for all levels of diabetes progression in Insulin and Metformin
insulin = pd.crosstab(administeredMeds.max_glu_serum, administeredMeds.insulin, normalize='index')
insulin = insulin.reindex(['Norm', '>200', '>300'])
metformin = pd.crosstab(administeredMeds.max_glu_serum, administeredMeds.metformin, normalize='index')
metformin = metformin.reindex(['Norm', '>200', '>300'])
# Plotting data
fig, (ax1, ax2) = plt.subplots(1, 2)
fig.suptitle('Insulin and Metformin Usage by Glucose Level')
insulin.plot.bar(stacked=True, ax=ax1)
ax1.legend(title='Insulin', bbox_to_anchor=(1, 1))
ax1.set_xlabel('Glucose Level')
ax1.set_ylabel('Proportion')
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=0)
metformin.plot.bar(stacked=True, ax=ax2)
ax2.legend(title='Metformin', bbox_to_anchor=(1, 1))
ax2.set_xlabel('Glucose Level');
ax2.set_ylabel('Proportion')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=0)
plt.tight_layout()

In [ ]:
# Subsetting data for readmission analysis
readmit = diabeticDataDf.loc[diabeticDataDf['readmitted'] != 'None']
readmit['readmitted'].value_counts()
df_melted_re_interest = pd.melt(readmit, id_vars=['readmitted'], value_vars=drugs_of_interest_columns)
df_melted_re_interest = df_melted_re_interest[df_melted_re_interest['value'] != 'No']
# Calculateing the proportions of readmission for each drug
proportion_data = df_melted_re_interest.groupby(['variable', 'readmitted']).size().unstack().fillna(0)
proportion_data = proportion_data.divide(proportion_data.sum(axis=1), axis=0)
# Plotting data
plt.figure(figsize=(8, 5))
# Settng the bottom of the bars
bottom = np.zeros(len(proportion_data))
# Iterate over each glucose level
for readmitted in ['NO', '>30', '<30']:
    plt.bar(
        proportion_data.index,
        proportion_data[readmitted],
        bottom=bottom,
        label=readmitted
    )
    bottom += proportion_data[readmitted].values
plt.title('Proportions of Select Drug Treatments by Readmittance')
plt.xlabel('Drug')
plt.ylabel('Proportion')
plt.xticks(ticks=range(len(proportion_data.index)), labels=[drug.capitalize() for drug in proportion_data.index],
           ha='center')
plt.legend(title='Readmittance Result', bbox_to_anchor=(1, 1))
plt.show()